## **Baseline Model Evaluation**

This notebook evaluates the winning solution from the [AICrowd Airborne Object Tracking Challenge](https://www.aicrowd.com/challenges/airborne-object-tracking-challenge) (2021) as the project baseline. The solution, developed by Dmytro Poplavskiy, uses a segmentation-based detection and tracking pipeline (SegTracker) with an ensemble of four deep learning models: HRNet32, HRNet48, GerNet-M with BiFPN, and DLA60.

Inference is performed across the entire custom hard dataset subset [outlined below](#dataset), with results presented both as an overall set of benchmark statistics and as performance metrics for each data subset.

**Evaluation Metrics:**
- **AFDR:** Airborne Flight Detection Rate. The primary detection metric.
- **FPPI:** False Positives Per Image. Must be ≤ 0.0005 to be ranked.
- **HFAR:** Horizontal False Alarm Rate. Must be ≤ 0.5 to be ranked.
- **EDR:** Encounter Detection Rate. Tracking-based metric across all horizon scenarios.

> ⚠️ ***Note:*** *Inference requires a CUDA-capable GPU. The `map_location` fix in `seg_tracker.py` allows CPU loading of weights for inspection, but full inference at acceptable speed requires GPU. Estimated runtime on GPU: ~2–3 FPS per flight sequence.*

In [1]:
# Import Required Libraries.
import json
import os
import random
import shutil
import sys
from glob import glob

import pandas as pd
import torch

# Because Life, Universe & Everything!
random.seed(42)

# Set Working Directory to Project Root.
os.chdir("..") 
PROJECT_ROOT = os.getcwd()

# Add Project Root to Path.
sys.path.insert(0, PROJECT_ROOT)

# Add Core Metric Folders to System Path.
sys.path.insert(0, os.path.join(PROJECT_ROOT, "core"))

# Import Local Project Funtions.
from logging_module import LogLevel, log
from src.baseline.seg_test import SegPredictor
from src.data.download_data import main as downloadData
from notebooks.helpers.notebook_helpers import get_next_run_dir, load_combined_gt, resolve_local_flights

# Confirm Project Root.
log(LogLevel.INFO, "baseline:imports", "Project Root Set: %s", PROJECT_ROOT)

<INFO ><baseline:imports> Project Root Set: /home/busatmd021/projects/2026/Vision-Based-Sense-and-Avoid



---

<br/>

## **Dataset**

The custom dataset used in this notebook is a curated hard subset of the full Airborne Object Tracking (AOT) dataset, specifically designed to stress-test model performance under the most demanding real-world conditions. Flights were selected to isolate encounters that represent the greatest challenge for vision-based detection and tracking systems.

**Isolation Criteria - encounters were selected for:**
- **Small Targets:** Airborne objects occupying only a few pixels per frame.
- **Multiple Targets:** Scenes containing more than one airborne object simultaneously.
- **Poor Visibility:** Flights affected by sun glare, adverse lighting, or challenging camera angles.

Together these conditions reflect the edge cases where sense-and-avoid systems are most likely to fail, and where reliable early detection is most critical for safe autonomous flight.

### **Airborne Object Tracking (AOT) - Hard Subset Summary**

| Metric                   | Value   | Description                                                             |
|--------------------------|--------:|-------------------------------------------------------------------------|
| **Total Flights**        | **1618**| **Total number of flight sequences in the dataset**                     |
| Small Object Flights     | 662     | Flights containing at least one object with bounding box area ≤ 36.0px².|
| Below Horizon Flights    | 1053    | Flights containing at least one object at or below the horizon.         |
| Multi-Object Flights     | 627     | Flights containing more than one airborne object in at least one frame. |
| No Object Frames         | 354     | Flights containing at least one frame with no airborne object present.  |
| *Avg. Sequence Length*   | *17.77s*| *Average duration of each flight sequence.*                             |

In [ ]:
# Set Path to Desired Evaluation Dataset.
DATASET_PATH = "configs/datasets/hard_dataset_flights.txt"
DATASET_PARTS = ["part1", "part2", "part3"]

# Set Results Variables (Auto-Incrementing Run Directory).
RESULTS_DIR = "outputs/results/baseline/"
run_name = get_next_run_dir(RESULTS_DIR)

# Evaluation Filtering.
MIN_SCORE = 0.985
MIN_TRACK_LEN = 30

# Evaluation Output Folders.
EVAL_GT_DIR = os.path.join(RESULTS_DIR, run_name, "gt")
EVAL_RESULT_DIR  = os.path.join(RESULTS_DIR, run_name, "result")
EVAL_SUMMARY_DIR = os.path.join(RESULTS_DIR, run_name, "summaries")

# Print Summary Statement.
log(LogLevel.INFO, "baseline:config", "Configuration Loaded.")
print(f"  Dataset Path  : {DATASET_PATH}")
print(f"  Dataset Parts : {DATASET_PARTS}")
print(f"  Run Name      : {run_name}")
print(f"  Results Path  : {RESULTS_DIR}")
print(f"  Min Score     : {MIN_SCORE}")
print(f"  Min Track Len : {MIN_TRACK_LEN}")

<INFO ><baseline:config> Configuration Loaded.
  Dataset Path  : configs/datasets/selected_flights.txt
  Dataset Parts : ['part1', 'part2', 'part3']
  Run Name      : run2
  Results Path  : outputs/results/baseline/
  Min Score     : 0.0
  Min Track Len : 0


In [ ]:
# Download the Selected Flight Sequences.
downloadData(DATASET_PATH)


---

<br/>

## **Data Preparation and Preprocessing**

Before inference can begin, the dataset must be resolved from the flight ID list down to what is actually available on disk, and the ground truth must be filtered to match. This cell performs two steps:

**1. Flight Resolution:** Parses the `.txt` flight ID file produced by `data_builder.py`, cross-references each entry against the local `data/<part>/Images/` directories, and builds a lookup of every flight that is both listed and downloaded. Flights in the list that have not been downloaded are logged and skipped.

**2. Ground Truth Filtering:** Loads the `groundtruth.json` file from each part's `ImageSets/` folder and merges them into a single structure, retaining only the entries that correspond to locally available flights. Entries for flights not on disk are silently skipped - this is expected, as the ground truth files reference the full AOT dataset rather than just the downloaded subset.

In [4]:
# Find all Available Sequences from the Provided Dataset.
local_flights = resolve_local_flights(DATASET_PATH, PROJECT_ROOT)

# Add Spacing in Console Output for Readability.
print()

# Log How Many Flights were Found & are Ready for Inference.
log(LogLevel.INFO, "baseline:preprocessing", "Flights Ready for Inference: %d", len(local_flights))

# Preview the First 5 Flights to Confirm Dataset is Correct.
for name, info in list(local_flights.items())[:5]:
    # Count No. Images in the Flight Directory.
    n_imgs = len(list(info["images_dir"].glob("*.png")))

    # Print Flight Part, Name, & No. Images.
    print(f"  [{info['part']}]  {name}  ({n_imgs} images)")

# If there are More than 5 Flights, Log How Many Additional Ones Exist.
if len(local_flights) > 5:
    log(LogLevel.DEBUG, "baseline:preprocessing", " ... and %d more.", len(local_flights) - 5)

# Load & Filter Ground Truth to Include Local Flights Only. 
combined_gt = load_combined_gt(DATASET_PARTS, local_flights, PROJECT_ROOT)
log(LogLevel.INFO, "baseline:preprocessing", "Flights in Filtered Ground Truth: %d", len(combined_gt['samples']))

# Save the Filtered Ground Truth to the Evaluation Folder.
os.makedirs(EVAL_GT_DIR, exist_ok=True)
gt_out_path = os.path.join(EVAL_GT_DIR, "groundtruth.json")

# Copy the Ground Truth Values to the .json.
with open(gt_out_path, "w") as f:
    json.dump(combined_gt, f)

# Print Success Message.
log(LogLevel.INFO, "baseline:preprocessing", "Ground Truth Saved: %s", gt_out_path)

<INFO ><notebook_helpers:resolve_flights> 6 Flights on Disk. 0 Not Found. (Run the Download Cell to Fetch Missing Flights).

<INFO ><baseline:preprocessing> Flights Ready for Inference: 6
  [part1]  part16e8de53f981f481c8090c7bcb9f66b44  (114 images)
  [part1]  part10f7714af364842449ee42f31b0446e2f  (120 images)
  [part2]  part2bf16dfb688224588b45cdbaf9013b315  (104 images)
  [part2]  part28fbfb91318f745138f01cf3ef429e560  (114 images)
  [part3]  part3aece5734256b483da5477a3ca4404c7e  (128 images)
<DEBUG><baseline:preprocessing>  ... and 1 more.

<INFO ><notebook_helpers:load_gt> 6 kept / 4154 total.
<INFO ><baseline:preprocessing> Flights in Filtered Ground Truth: 6
<INFO ><baseline:preprocessing> Ground Truth Saved: outputs/results/baseline/run2/gt/groundtruth.json



---

<br/>

## **Evaluate Baseline Model**

### **Run Baseline Inference**

**1.** Prepare evaluation directory and configure environment variables.

In [5]:
# Create a Directory where all Evaluation Flight Data will be Stored.
EVAL_DATA_DIR = os.path.join(RESULTS_DIR, run_name, "data")
os.makedirs(EVAL_DATA_DIR, exist_ok=True)

# Link Each Local Flight's Image Folder into the Evaluation Directory.
for folder_name, info in local_flights.items():
    link_path = os.path.join(EVAL_DATA_DIR, folder_name)
    if not os.path.exists(link_path):
        os.symlink(info["images_dir"], link_path)

# Set Environment Variables that AirbornePredictor and SegPredictor Read.
os.environ["TEST_DATASET_PATH"] = EVAL_DATA_DIR
os.environ["INFERENCE_OUTPUT_PATH"] = RESULTS_DIR
os.environ["DATASET_ENV"] = run_name
os.environ["MIN_TRACK_LEN"] = str(MIN_TRACK_LEN)
os.environ["MIN_SCORE"] = str(MIN_SCORE)

# Limit Inference to Only the Specified Flights.
os.environ["PARTIAL_RUN_FLIGHTS"] = ",".join(local_flights.keys())

# Print Summary Statement.
log(LogLevel.INFO, "baseline:inference", "Environment Configured.")
print(f"  Test Data Path     : {EVAL_DATA_DIR}")
print(f"  Flights to Process : {len(local_flights)}")
print(f"  MIN_SCORE          : {MIN_SCORE}")
print(f"  MIN_TRACK_LEN      : {MIN_TRACK_LEN}")

<INFO ><baseline:inference> Environment Configured.
  Test Data Path     : outputs/results/baseline/run2/data
  Flights to Process : 6
  MIN_SCORE          : 0.0
  MIN_TRACK_LEN      : 0


**2.** Run segmentation predictor on all evaluation flights.

> ⚠️ ***This cells requires a CUDA GPU and will take several hours to complete.*** *Skip this cell if inference has already been run and results are present in `RESULTS_DIRS`.*

In [6]:
# Check GPU is CUDA Enabled.
if torch.cuda.is_available():
    log(LogLevel.INFO, "baseline:device_check", "CUDA-Enabled GPU is Available.")
    print(f"  GPU Name: {torch.cuda.get_device_name(0)}.")
    print(f"  CUDA Version: {torch.version.cuda}.")
else:
      log(LogLevel.WARNING, "baseline:device_check", "No CUDA-Enabled GPU Found, Using CPU Instead.")

<INFO ><baseline:device_check> CUDA-Enabled GPU is Available.
  GPU Name: NVIDIA GeForce RTX 5060 Ti.
  CUDA Version: 12.8.


In [7]:
# Initialise Predictor (Reads Env Variables).
predictor = SegPredictor()

# Run Inference Over all Flights.
predictor.run()

# Print Summary Statement.
log(LogLevel.INFO, "baseline:inference", "Inference Complete.")
print(f"  Raw Results Saved to: {os.path.join(RESULTS_DIR, run_name)}")

# Set Results Directories. 
RAW_RESULTS_PATH = os.path.join(RESULTS_DIR, run_name, "result.json")
METRICS_RESULTS_PATH = os.path.join(EVAL_RESULT_DIR, "result.json")

# Ensure Output Directory Exists.
os.makedirs(EVAL_RESULT_DIR, exist_ok=True)

# Copy the Results File.
shutil.copy2(RAW_RESULTS_PATH, METRICS_RESULTS_PATH)

# Print Summary Statement.
log(LogLevel.INFO, "baseline:inference", "Results Copied to Metrics Directory.")
print(f"  Metrics-Ready Results Stored at : {METRICS_RESULTS_PATH}")

hrnet_w32 upscale: nearest comb outputs: 512
Feature channels: [32, 64, 128, 256]
Using custom backbone gernet_m
{'name': 'tf_efficientdet_d2', 'backbone_name': 'gernet_m', 'backbone_args': {'in_chans': 2}, 'backbone_indices': None, 'image_size': [512, 512], 'num_classes': 2, 'min_level': 3, 'max_level': 7, 'num_levels': 5, 'num_scales': 3, 'aspect_ratios': [[1.0, 1.0], [1.4, 0.7], [0.7, 1.4]], 'anchor_scale': 4.0, 'pad_type': 'same', 'act_type': 'swish', 'norm_layer': None, 'norm_kwargs': {'eps': 0.001, 'momentum': 0.01}, 'box_class_repeats': 3, 'fpn_cell_repeats': 5, 'fpn_channels': 112, 'separable_conv': True, 'apply_resample_bn': True, 'conv_after_downsample': False, 'conv_bn_relu_pattern': False, 'use_native_resize_op': False, 'downsample_type': 'max', 'upsample_type': 'nearest', 'redundant_bias': True, 'head_bn_level_first': False, 'head_act_type': None, 'fpn_name': None, 'fpn_config': None, 'fpn_drop_path_rate': 0.0, 'alpha': 0.25, 'gamma': 1.5, 'label_smoothing': 0.0, 'legacy_f

100%|██████████| 128/128 [01:04<00:00,  1.98it/s]

<INFO ><baseline:inference> Inference Complete.
  Raw Results Saved to: outputs/results/baseline/run2
<INFO ><baseline:inference> Results Copied to Metrics Directory.
  Metrics-Ready Results Stored at : outputs/results/baseline/run2/result/result.json


### **Run Metrics Evaluation**

Run the official AICrowd evaluation metrics against the ground truth for each dataset part, saving per-part summaries to `SUMMARIES_DIR`.

In [8]:
# Locate the Competition Metrics Script.
metrics_script = os.path.join(PROJECT_ROOT, "core/metrics/run_airborne_metrics.py")

# Call the Competition Metrics Script.
!python3 {metrics_script} \
    --dataset-folder   {EVAL_GT_DIR} \
    --results-folder   {EVAL_RESULT_DIR} \
    --summaries-folder {EVAL_SUMMARY_DIR} \
    --min-score        {MIN_SCORE} \
    --min-track-len    {MIN_TRACK_LEN}

# Print Message on Completion. 
log(LogLevel.INFO, "baseline:evaluation", "Metrics Complete.")

2026-03-28 00:12:05,868:INFO:run_airborne_metrics.py:160 Encounter ground truth: outputs/results/baseline/run2/gt/groundtruth_with_encounters_maxRange700_maxGap3_minEncLen30.csv
2026-03-28 00:12:05,869:INFO:calculate_encounters.py:83 Asserting outputs/results/baseline/run2/gt/groundtruth.json format
2026-03-28 00:12:05,870:INFO:pandas_utils.py:87 Reading ground truth
2026-03-28 00:12:05,870:INFO:pandas_utils.py:61 Reading provided outputs/results/baseline/run2/gt/groundtruth.json
2026-03-28 00:12:05,870:INFO:pandas_utils.py:68 Loading .json
2026-03-28 00:12:05,885:INFO:pandas_utils.py:75 Normalizing json. This operation is time consuming. The result .csv will be saved Please consider providing .csv file next time
2026-03-28 00:12:05,971:INFO:calculate_encounters.py:268 Saving groundtruth in .csv format, please use .csv in the future
2026-03-28 00:12:06,010:INFO:calculate_encounters.py:272 Filtering ground truth to get intruders in the specified range <= 700m.
2026-03-28 00:12:06,013:IN

### **Baseline Benchmark Summary**

Display the aggregated statistics across all three parts of the dataset in order to provide an overall baseline performance for the full hard dataset. Additionally, statistics for each individual subset of the hard dataset are also displayed for more detailed analysis.

In [9]:
# Find the Summary JSON in the Summaries Directory.
summary_files = glob(os.path.join(EVAL_SUMMARY_DIR, "*.json"))

# Check that the Summary Files Exist.
if not summary_files:
    log(LogLevel.ERROR, "baseline:summary", "No Summary Files Found! Run Inference First.")
else:
    # Load the Summary JSON.
    with open(summary_files[0]) as f:
        summary = json.load(f)

    # Extract Key Metrics from Summary JSON.
    score         = summary["min_det_score"]
    fppi          = summary["fppi"]
    hfar          = summary["far"]
    afdr          = summary["fl_dr_in_range"]
    det_aircraft  = summary["det_aircraft_in_range"]
    tot_aircraft  = summary["tot_aircraft_in_range"]
    edr_all       = summary["Detection"]["Encounters"]["300"]["All"]["dr"]
    edr_above     = summary["Detection"]["Encounters"]["300"]["Above Horizon"]["dr"]
    edr_below     = summary["Detection"]["Encounters"]["300"]["Below Horizon"]["dr"]

    # Rank Metrics Against Thresholds.
    fppi_status = "PASS (≤ 0.0005)" if fppi  <= 0.0005 else f"FAIL (> 0.0005)"
    hfar_status = "PASS (≤ 0.5)"    if hfar  <= 0.5    else f"FAIL (> 0.0005)"

    # Build Summary DataFrame.
    summary_df = pd.DataFrame([
        {"Metric": "Detection Score",                       "Value": f"{score:.4f}",       "Status": ""},
        {"Metric": "AFDR (Airborne Flight Detection Rate)", "Value": f"{afdr:.4f}",        "Status": ""},
        {"Metric": "FPPI (False Positives Per Image)",      "Value": f"{fppi:.6f}",        "Status": fppi_status},
        {"Metric": "HFAR (Horizontal False Alarm Rate)",    "Value": f"{hfar:.4f}",        "Status": hfar_status},
        {"Metric": "EDR - All Scenarios",                   "Value": f"{edr_all:.4f}",     "Status": ""},
        {"Metric": "EDR - Above Horizon",                   "Value": f"{edr_above:.4f}",   "Status": ""},
        {"Metric": "EDR - Below Horizon",                   "Value": f"{edr_below:.4f}",   "Status": ""},
        {"Metric": "Total Aircraft Detected",               "Value": f"{det_aircraft}",    "Status": ""},
        {"Metric": "Total Aircraft in Dataset",             "Value": f"{tot_aircraft}",    "Status": ""},
    ])

    # Print Summary to Console.
    print("\n" + "=" * 60)
    print(" BASELINE MODEL - FULL HARD DATASET BENCHMARK SUMMARY")
    print(" Model: Dmytro Poplavskiy Winning Solution (AICrowd 2021)")
    print("=" * 60)
    display(summary_df.set_index("Metric"))

    # Save Aggregated Summary to CSV.
    output_path = os.path.join(EVAL_SUMMARY_DIR, "baseline_benchmark_summary.csv")
    summary_df.to_csv(output_path, index=False)
    print()
    log(LogLevel.INFO, "baseline:summary", "Summary Saved to: %s", output_path)


 BASELINE MODEL - FULL HARD DATASET BENCHMARK SUMMARY
 Model: Dmytro Poplavskiy Winning Solution (AICrowd 2021)


,Value,Status
Metric,,
Detection Score,0.8325,
AFDR (Airborne Flight Detection Rate),0.0000,
FPPI (False Positives Per Image),0.056583,FAIL (> 0.0005)
HFAR (Horizontal False Alarm Rate),20.0000,FAIL (> 0.0005)
EDR - All Scenarios,0.0000,
EDR - Above Horizon,0.0000,
EDR - Below Horizon,0.0000,
Total Aircraft Detected,0,
Total Aircraft in Dataset,704,



<INFO ><baseline:summary> Summary Saved to: outputs/results/baseline/run2/summaries/baseline_benchmark_summary.csv



---

<br/>

## **Hard Dataset Results**

The table below reports baseline performance broken down by scenario category across the custom
hard dataset (1,618 flights). Flights were selected from the full AOT dataset using configurable
filter criteria to isolate the most challenging sequences, providing a more targeted benchmark
than the full competition dataset.

| Scenario                     | Flights | FPPI | AFDR | HFAR | EDR |
|------------------------------|---------|------|------|------|-----|
| **Overall (All Criteria)**   |**1,618**| —    | —    | —    | —   |
| Small Object (bbox ≤ 36 px²) | 662     | —    | —    | —    | —   |
| Below Horizon                | 1,053   | —    | —    | —    | —   |
| Multi-Object                 | 627     | —    | —    | —    | —   |
| No Visible Object            | 354     | —    | —    | —    | —   |

*Results evaluated at a minimum detection confidence of 0.985 and a minimum track length of 30 frames (≈ 3 seconds at 10 fps).*

In [ ]:
def evaluate_subset(
    subset_txt      : str,
    run_results_dir : str,
    run_gt_dir      : str,
    metrics_script  : str,
    dataset_parts   : list,
    project_root    : str,
) -> None:
    """
    Re-evaluates metrics on a subset of flights without re-running inference.

    Filters the existing result.json and groundtruth.json to only include
    flights listed in the subset .txt file, then runs the competition metrics
    script on the filtered data and displays the summary statistics.

    Args:
        subset_txt      (str):  Path to the .txt file listing subset flight IDs.
        run_results_dir (str):  Path to the run's result directory (e.g. outputs/results/baseline/run0/result).
        run_gt_dir      (str):  Path to the run's GT directory (e.g. outputs/results/baseline/run0/gt).
        metrics_script  (str):  Path to run_airborne_metrics.py.
        dataset_parts   (list): List of dataset parts e.g. ["part1", "part2", "part3"].
        project_root    (str):  Absolute path to the project root.
    """
    # Derive Subset Name from the .txt Filename for Output Folder Naming.
    subset_name = os.path.splitext(os.path.basename(subset_txt))[0]

    # Set Output Directories for this Subset Evaluation.
    subset_base        = os.path.join(os.path.dirname(run_results_dir), "subsets", subset_name)
    subset_result_dir  = os.path.join(subset_base, "result")
    subset_gt_dir      = os.path.join(subset_base, "gt")
    subset_summary_dir = os.path.join(subset_base, "summaries")

    os.makedirs(subset_result_dir,  exist_ok=True)
    os.makedirs(subset_gt_dir,      exist_ok=True)
    os.makedirs(subset_summary_dir, exist_ok=True)

    log(LogLevel.INFO, "baseline:subset", "Evaluating Subset: %s", subset_name)

    # Resolve the Subset Flight IDs from the .txt File.
    subset_flights = resolve_local_flights(subset_txt, project_root)
    subset_ids     = set(subset_flights.keys())

    log(LogLevel.INFO, "baseline:subset", "Subset Contains %d Flights.", len(subset_ids))

    # -------------------------------------------------------------------------
    #  Filter result.json to Subset Flights Only.
    # -------------------------------------------------------------------------
    full_result_path   = os.path.join(run_results_dir, "result.json")
    subset_result_path = os.path.join(subset_result_dir, "result.json")

    with open(full_result_path) as f:
        full_results = json.load(f)

    # Keep Only Detections Belonging to Subset Flights.
    subset_results = [
        entry for entry in full_results
        if any(flight_id in entry.get("img_name", "") for flight_id in subset_ids)
    ]

    with open(subset_result_path, "w") as f:
        json.dump(subset_results, f)

    log(LogLevel.INFO, "baseline:subset",
        "Filtered Results: %d / %d Entries.", len(subset_results), len(full_results))

    # -------------------------------------------------------------------------
    #  Filter groundtruth.json to Subset Flights Only.
    # -------------------------------------------------------------------------
    full_gt_path   = os.path.join(run_gt_dir, "groundtruth.json")
    subset_gt_path = os.path.join(subset_gt_dir, "groundtruth.json")

    with open(full_gt_path) as f:
        full_gt = json.load(f)

    # Filter GT Samples to Subset Flights Only.
    subset_gt = {
        "samples": {
            flight_id: flight_data
            for flight_id, flight_data in full_gt["samples"].items()
            if flight_id in subset_ids
        }
    }

    with open(subset_gt_path, "w") as f:
        json.dump(subset_gt, f)

    log(LogLevel.INFO, "baseline:subset",
        "Filtered GT: %d / %d Samples.", len(subset_gt["samples"]), len(full_gt.get("samples", [])))

    # -------------------------------------------------------------------------
    #  Run Competition Metrics on the Filtered Subset.
    # -------------------------------------------------------------------------
    log(LogLevel.INFO, "baseline:subset", "Running Metrics on Subset.")

    !python3 {metrics_script} \
        --dataset-folder   {subset_gt_dir} \
        --results-folder   {subset_result_dir} \
        --summaries-folder {subset_summary_dir}

    # -------------------------------------------------------------------------
    #  Load and Display Summary Statistics.
    # -------------------------------------------------------------------------
    summary_files = glob(os.path.join(subset_summary_dir, "*.json"))

    if not summary_files:
        log(LogLevel.ERROR, "baseline:subset", "No Summary Files Found.")
        return

    with open(summary_files[0]) as f:
        summary = json.load(f)

    # Extract Key Metrics.
    fppi         = summary["fppi"]
    hfar         = summary["far"]
    afdr         = summary["fl_dr_in_range"]
    det_aircraft = summary["det_aircraft_in_range"]
    tot_aircraft = summary["tot_aircraft_in_range"]
    edr_all      = summary["Detection"]["Encounters"]["300"]["All"]["dr"]
    edr_above    = summary["Detection"]["Encounters"]["300"]["Above Horizon"]["dr"]
    edr_below    = summary["Detection"]["Encounters"]["300"]["Below Horizon"]["dr"]

    # Rank Against Thresholds.
    fppi_status = "PASS (≤ 0.0005)" if fppi <= 0.0005 else f"FAIL — Actual: {fppi:.6f}"
    hfar_status = "PASS (≤ 0.5)"    if hfar <= 0.5    else f"FAIL — Actual: {hfar:.4f}"

    # Build Summary DataFrame.
    summary_df = pd.DataFrame([
        {"Metric": "AFDR (Airborne Flight Detection Rate)", "Value": f"{afdr:.4f}",      "Status": ""},
        {"Metric": "FPPI (False Positives Per Image)",      "Value": f"{fppi:.6f}",      "Status": fppi_status},
        {"Metric": "HFAR (Horizontal False Alarm Rate)",    "Value": f"{hfar:.4f}",      "Status": hfar_status},
        {"Metric": "EDR - All Scenarios",                   "Value": f"{edr_all:.4f}",   "Status": ""},
        {"Metric": "EDR - Above Horizon",                   "Value": f"{edr_above:.4f}", "Status": ""},
        {"Metric": "EDR - Below Horizon",                   "Value": f"{edr_below:.4f}", "Status": ""},
        {"Metric": "Total Aircraft Detected",               "Value": f"{det_aircraft}",  "Status": ""},
        {"Metric": "Total Aircraft in Dataset",             "Value": f"{tot_aircraft}",  "Status": ""},
    ])

    # Display Summary.
    print("\n" + "=" * 60)
    print(f" SUBSET EVALUATION SUMMARY — {subset_name.upper()}")
    print("=" * 60)
    display(summary_df.set_index("Metric").style
        .applymap(lambda v: "color: green" if v and "PASS" in str(v) else
                            "color: red"   if v and "FAIL" in str(v) else "",
                  subset=["Status"])
    )

    # Save Summary CSV.
    output_path = os.path.join(subset_summary_dir, f"{subset_name}_summary.csv")
    summary_df.to_csv(output_path, index=False)
    log(LogLevel.INFO, "baseline:subset", "Summary Saved to: %s", output_path)



evaluate_subset(
  subset_txt="configs/datasets/hard_subsets/multi_object_flights.txt",
  run_results_dir="outputs/results/baseline/run0/result",
  run_gt_dir="outputs/results/baseline/run0/gt",
  metrics_script=os.path.join(PROJECT_ROOT, "core/metrics/run_airborne_metrics.py"),
  dataset_parts= ["part1", "part2", "part3"],
  project_root=PROJECT_ROOT
)

---